In [0]:
import pyspark
import pandas as pd

input_folder = '../data/raw'

# Read JSON files using pandas, then convert to Spark DataFrames due to limitations of free Databricks 
df_offers = spark.createDataFrame(pd.read_json(f'{input_folder}/offers.json'))
df_profile = spark.createDataFrame(pd.read_json(f'{input_folder}/profile.json'))
df_trans = spark.createDataFrame(pd.read_json(f'{input_folder}/transactions.json'))

# Conjunto de transações

In [0]:
# In df_trans, the column 'value' is a struct. Let's convert it to 4 columns, one for each attribute
from pyspark.sql.functions import col

value_attrs = df_trans.schema['value'].dataType.names
df_trans_expanded = df_trans
for attr in value_attrs:
    df_trans_expanded = df_trans_expanded.withColumn(attr, col('value')[attr])
df_trans_expanded = df_trans_expanded.drop('value')
display(df_trans_expanded.head())


Existem duas colunas relacionadas ao id da oferta e minha hipótese é que elas devam se referir à mesma coisa.

In [0]:
# Offer id and Offer_id seems to be the same collumn. Let's analyse if there is at least a single record with distinct values in both collumns.
from pyspark.sql.functions import col, isnotnull, count

total_count = df_trans_expanded.count()

offer_id_notna_count = df_trans_expanded.filter(col('offer_id').isNotNull()).count()
offer_id_space_notna_count = df_trans_expanded.filter(col('offer id').isNotNull()).count()
both_notna_count = df_trans_expanded.filter(col('offer_id').isNotNull() & col('offer id').isNotNull()).count()

print((offer_id_notna_count / total_count) * 100, '% de offer_id não nulos.')
print((offer_id_space_notna_count / total_count) * 100, '% de offer id não nulos.')
print((both_notna_count / total_count) * 100, '% de ambos não nulos.')

Como não há nenhum registro em que existe dois valores distintos para offer_id e offer id, estou assumindo que esse foi um erro de entrada no arquivo que acabou criando um atributo reduntante. Logo, pode ser aglutinado em um só.

In [0]:
from pyspark.sql.functions import coalesce

df_trans_expanded = df_trans_expanded.withColumn(
    'offer_id',
    coalesce(col('offer_id'), col('offer id'))
).drop('offer id')

In [0]:
# Checking if the fusion operation was successful
total_count = df_trans_expanded.count()
offer_id_notna_count = df_trans_expanded.filter(col('offer_id').isNotNull()).count()

print(offer_id_notna_count / total_count * 100)

Antes da operação de mesclagem, tínhamos 43% de dados em uma das colunas e 11% em outra, sem sobreposição. Após a operação, ficamos com 54% de dados preenchidos na coluna resultante, o que indica que deu certo. Agora que o conjunto de transações está ajustado, podemos agregá-lo aos outros conjunto de dados.

In [0]:
df = df_trans_expanded.join(df_offers, df_trans_expanded.offer_id == df_offers.id, 'inner').drop('id')
df = df.join(df_profile, df_trans_expanded.account_id == df_profile.id, 'inner').drop('id')
display(df)

Podemos fazer mais algumas limpezas nos dados como ajustar o campo de data registered_on que está como inteiro, segmentar o campo channel em colunas booleanas, aplicando One-Hot Encoding nele e tratar outras colunas categóricas, além de tratar missing data também.

# Limpeza de dados

In [0]:
# Adjust date field
from pyspark.sql import functions as F
df = df.withColumn('registered_on', F.to_date(F.col('registered_on').cast('string'), 'yyyyMMdd'))

In [0]:
from pyspark.sql.functions import array_contains

# Get distinct channel values
distinct_channels = [row.channel for row in df.selectExpr("explode(channels) as channel").distinct().collect()]

# Create one-hot encoded columns
for channel in distinct_channels:
    df = df.withColumn(f'channel_{channel}', array_contains(df['channels'], channel))

In [0]:
display(df.select('gender').distinct())
display(df.select('event').distinct())
display(df.select('offer_type').distinct())

from pyspark.sql.functions import when

# Categorical encoding for 'gender'
df = df.withColumn('gender_num', 
                   when(df.gender == 'M', 1)
                   .when(df.gender == 'F', 2)
                   .when(df.gender == 'O', 3)
                   .otherwise(0))

# Categorical encoding for 'event'
event_mapping = {'transaction': 1, 'offer received': 2, 'offer viewed': 3, 'offer completed': 4}
df = df.withColumn('event_num', 
                   when(df.event == 'transaction', 1)
                   .when(df.event == 'offer received', 2)
                   .when(df.event == 'offer viewed', 3)
                   .when(df.event == 'offer completed', 4)
                   .otherwise(0))

# Categorical encoding for 'offer_type'
offer_type_mapping = {'bogo': 1, 'discount': 2, 'informational': 3}
df = df.withColumn('offer_type_num', 
                   when(df.offer_type == 'bogo', 1)
                   .when(df.offer_type == 'discount', 2)
                   .when(df.offer_type == 'informational', 3)
                   .otherwise(0))

In [0]:
from pyspark.sql.functions import col, sum as Fsum, lit

# Show count of nulls per column as two columns: column name and number of nulls
null_counts = [(c, df.filter(col(c).isNull()).count()) for c in df.columns]
null_counts_df = spark.createDataFrame(null_counts, ['column_name', 'null_count'])
display(null_counts_df)

df = df.fillna({'amount': 0, 'reward': 0, 'credit_card_limit': 0, 'gender': 'N'})

# Show count of nulls per column as two columns: column name and number of nulls
null_counts = [(c, df.filter(col(c).isNull()).count()) for c in df.columns]
null_counts_df = spark.createDataFrame(null_counts, ['column_name', 'null_count'])
display(null_counts_df)

In [0]:
import pandas as pd

output_folder = '../data/processed/data.parquet'
# Convert Spark DataFrame to pandas and save due to limitations of free Databricks 
pdf = pd.DataFrame(df.collect(), columns=df.columns)
pdf.to_parquet(output_folder)